<a href="https://colab.research.google.com/github/veer0110/Agentic-AI/blob/main/travel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ⚠️  IMPORTANT — Read before running
# After this cell finishes, go to:  Runtime → Restart Runtime
# Then run all cells again from the top (Runtime → Run All)
# This is required in Google Colab whenever new packages are installed.

!pip install -q -U \
    "langchain>=0.3.7,<0.4.0" \
    "langchain-groq>=0.2.0" \
    "langchain-community>=0.3.7" \
    "langchain-core>=0.3.7" \
    duckduckgo-search \
    requests

# Verify installed versions
import langchain, langchain_groq, langchain_community
print(f'✅ langchain          : {langchain.__version__}')
print(f'✅ langchain-groq     : {langchain_groq.__version__}')
print(f'✅ langchain-community: {langchain_community.__version__}')
print()
print('⚠️  NOW: Runtime → Restart Runtime → then run all cells again.')

✅ langchain          : 0.3.30
✅ langchain-groq     : 0.3.8
✅ langchain-community: 0.3.31

⚠️  NOW: Runtime → Restart Runtime → then run all cells again.


In [ ]:
import os, requests, getpass

# LangChain — Core
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

# LangChain — Groq LLM
from langchain_groq import ChatGroq

# NOTE: We intentionally do NOT use AgentExecutor.
# We write a simple manual agent loop instead — more reliable with Groq
# and more educational (you see exactly how the loop works).

# LangChain — Memory (for Session 2)
from langchain_community.chat_message_histories import ChatMessageHistory

# DuckDuckGo Search
from duckduckgo_search import DDGS

import langchain
print(f'✅ All libraries imported  |  LangChain {langchain.__version__}')

✅ All libraries imported  |  LangChain 0.3.30


In [ ]:
# Securely enter your API keys — not stored, session-only
GROQ_API_KEY = getpass.getpass('🔑 Enter your Groq API Key: ')
OPENWEATHER_API_KEY = getpass.getpass('🌤️  Enter your OpenWeatherMap API Key: ')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

print('✅ API keys configured!')

🔑 Enter your Groq API Key: ··········
🌤️  Enter your OpenWeatherMap API Key: ··········
✅ API keys configured!


In [ ]:
# ── MODEL CHOICE ────────────────────────────────────────────
GROQ_MODEL = 'qwen/qwen3.6-27b'   # smaller, faster, same tool support
    # GROQ_MODEL = 'llama-3.3-70b-versatile'
llm = ChatGroq(
        model=GROQ_MODEL,
        temperature=0,
        max_tokens=2048,
        streaming=False,      # must be False for reliable tool calling
        api_key=GROQ_API_KEY
    )
response = llm.invoke('Give one travel fact about India in one sentence.')
print(f'✅ LLM ready — model: {GROQ_MODEL}')
print('🤖', response.content)

✅ LLM ready — model: qwen/qwen3.6-27b
🤖 
<think>
Thinking Process:

1.  **Deconstruct the user's request:**
    *   Topic: Travel fact about India.
    *   Quantity: One fact.
    *   Format: One sentence.

2.  **Brainstorm potential facts:**
    *   India has the Taj Mahal. (Too cliché?)
    *   India has the world's largest democracy. (Political, not strictly travel-focused, though related.)
    *   India has the highest railway crossing in the world. (Good, specific.)
    *   India has the longest railway platform. (Good, specific.)
    *   India has the only place where you can see the sun rise and set from the same spot? (No, that's not unique or accurate.)
    *   India has the world's largest animal migration? (No, that's usually Serengeti, though India has bird migrations.)
    *   India has the only place where you can see the Himalayas and the Indian Ocean? (Kerala? Maybe, but not a "fact" per se.)
    *   India has the world's largest postal network? (Interesting, but maybe 

In [ ]:
# ── BUILD YOUR FIRST CHAIN ──────────────────────────────────

# Step A: Prompt Template
prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful and enthusiastic travel assistant.'),
    ('human', '{input}')
])

# Step B: Output Parser — converts AIMessage object to plain string
parser = StrOutputParser()

# Step C: Assemble the Chain using the pipe operator
chain = prompt | llm | parser

print('✅ Chain built: prompt | llm | parser')

✅ Chain built: prompt | llm | parser


In [ ]:
# ── RUN THE CHAIN ───────────────────────────────────────────

response = chain.invoke({'input': 'What are the top 3 things to do in Kerala?'})
print('🤖 Travel Assistant:')
print(response)

🤖 Travel Assistant:

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "What are the top 3 things to do in Kerala?"
   - **Location:** Kerala, India
   - **Request:** Top 3 things to do
   - **Tone/Role:** Helpful and enthusiastic travel assistant

2.  **Identify Key Attractions/Experiences in Kerala:**
   Kerala is known for:
   - Backwaters (houseboat cruises, especially in Alleppey/Alleppey)
   - Hill stations & nature (Munnar tea plantations, wildlife, trekking)
   - Culture & wellness (Ayurveda, Kathakali dance, temples, beaches)
   - Wildlife (Periyar Tiger Reserve)
   - Food (Kerala cuisine, spice markets)

3.  **Select Top 3 (Balanced, Iconic, Diverse):**
   I need to pick 3 that represent the best of Kerala and offer different experiences:
   1. **Cruise the Backwaters on a Houseboat** (Iconic, unique to Kerala, relaxing)
   2. **Explore Munnar’s Tea Plantations & Hill Stations** (Nature, scenic, cultural)
   3. **Experience Ayurveda & Traditio

In [ ]:
# ── INSPECT EACH STEP SEPARATELY ────────────────────────────
# This makes the chain's internals visible

# What does the prompt look like after filling the template?
filled = prompt.invoke({'input': 'Top places in Kerala'})
print('📝 Filled Prompt (what the LLM actually receives):',
      str(filled)[:200], '...')
print()

# What does the raw LLM response look like before parsing?
raw = (prompt | llm).invoke({'input': 'Top places in Kerala'})
print('📦 Raw LLM response type:', type(raw).__name__)
print('   .content:', raw.content[:120], '...')


📝 Filled Prompt (what the LLM actually receives): messages=[SystemMessage(content='You are a helpful and enthusiastic travel assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Top places in Kerala', additional_kwargs={},  ...

📦 Raw LLM response type: AIMessage
   .content: 
<think>
Here's a thinking process:

1.  **Understand User Request:** The user is asking for "Top places in Kerala". Thi ...


In [ ]:
# ── PARAMETERISED PROMPT TEMPLATE ───────────────────────────

trip_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are an expert travel planner. '
     'Create a concise {days}-day itinerary for {destination} '
     'with a daily budget of {daily_budget} INR. '
     'Include must-visit places, local food, and practical tips.'),
    ('human', 'Plan my trip!')
])

trip_chain = trip_prompt | llm | StrOutputParser()
print('✅ Parameterised chain ready!')

✅ Parameterised chain ready!


In [ ]:
# ── SAME CHAIN, DIFFERENT DESTINATIONS ──────────────────────

print('=' * 55)
print('🏖️  GOA — 3 Days, ₹5,000/day')
print('=' * 55)
print(trip_chain.invoke({'destination': 'Goa', 'days': '3', 'daily_budget': '5000'}))

print()
print('=' * 55)
print('🏔️  MANALI — 4 Days, ₹4,000/day')
print('=' * 55)
print(trip_chain.invoke({'destination': 'Manali', 'days': '4', 'daily_budget': '4000'}))

🏖️  GOA — 3 Days, ₹5,000/day

<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Role:** Expert travel planner
   - **Destination:** Goa
   - **Duration:** 3 days
   - **Budget:** 5000 INR per day (total 15,000 INR)
   - **Requirements:** Must-visit places, local food, practical tips
   - **Format:** Concise itinerary

2.  **Deconstruct Requirements:**
   - **Daily Budget:** 5000 INR/day is quite comfortable for Goa if managed well. Covers accommodation, food, transport, activities, and miscellaneous.
   - **Duration:** 3 days/2 nights (typical for a short trip)
   - **Must-visit places:** Need a mix of beaches, heritage, culture, and nature. South Goa for tranquility, North Goa for vibe/heritage.
   - **Local food:** Seafood, Goan cuisine (fish curry rice, vindaloo, bebinca, feni, etc.)
   - **Practical tips:** Transport, booking, safety, money, best time, etc.
   - **Concise:** Keep it structured, bullet-point friendly, no fluff.

3.  **Budget Breakdown (per day ~

In [ ]:
@tool
def my_tool(input: str) -> str:
    """LLM reads this docstring to understand what the tool does."""
    return result


In [ ]:
# ── TOOL 1: WEB SEARCH ──────────────────────────────────────
# Keep the docstring SHORT — the LLM reads it to decide when to call the tool.
# Long docstrings can confuse the function-call generator on Groq.

@tool
def web_search(query: str) -> str:
    '''Search the web for travel destination info, attractions, and tips.
    Args: query — search string.
    Returns: top search results as text.
    '''
    try:
        results = []
        with DDGS() as ddgs:
            for r in list(ddgs.text(query, max_results=3)):
                results.append(f"- {r.get('title','')}\n  {r.get('body','')}")
        # Truncate to 1500 chars — avoids overflowing the context window
        output = '\n\n'.join(results)
        return output[:1500] + '...' if len(output) > 1500 else output
    except Exception as e:
        return f'Search unavailable: {str(e)}'

print(f'✅ Tool 1: {web_search.name}')

✅ Tool 1: web_search


In [ ]:
# ── TEST WEB SEARCH STANDALONE ──────────────────────────────

result = web_search.invoke({'query': 'best places to visit in Rajasthan India travel guide'})
print('🔍 Search Result (first 500 chars):')
print(result[:500], '...')

/tmp/ipykernel_2129/596202386.py:13: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


🔍 Search Result (first 500 chars):
 ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── TOOL 2: REAL-TIME WEATHER ───────────────────────────────

@tool
def get_weather(city: str) -> str:
    '''Get current weather for a city.
    Args: city — city name (e.g. Mumbai, Goa, Shimla).
    Returns: temperature, conditions, humidity, packing tip.
    '''
    try:
        resp = requests.get(
            'http://api.openweathermap.org/data/2.5/weather',
            params={'q': city, 'appid': OPENWEATHER_API_KEY, 'units': 'metric'},
            timeout=8
        )
        d = resp.json()
        if d.get('cod') == 200:
            cond  = d['weather'][0]['description'].capitalize()
            temp  = d['main']['temp']
            feels = d['main']['feels_like']
            hum   = d['main']['humidity']
            tip   = ('Pack light clothes!' if temp > 30 else
                     'Light layers recommended.' if temp > 20 else
                     'Carry a jacket.' if temp > 10 else
                     'Pack warm clothes!')
            return (f'Weather in {city}: {cond}, {temp}°C '
                    f'(feels {feels}°C), humidity {hum}%. Tip: {tip}')
        return f'Could not get weather for {city}. Check the city name.'
    except Exception as e:
        return f'Weather error: {str(e)}'

print(f'✅ Tool 2: {get_weather.name}')

✅ Tool 2: get_weather


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
result = get_weather.invoke({'city': 'Mumbai'})
print('🌤️ Current Weather:')
print(result)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🌤️ Current Weather:
Weather in Mumbai: Light rain, 30.99°C (feels 37.99°C), humidity 74%. Tip: Pack light clothes!


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
import re, json, warnings
warnings.filterwarnings('ignore')

def _parse_tool_call(text):
    m = re.search(r'<function=(\w+)(\{.*?\})\s*(?:</function>)?', text, re.DOTALL)
    if m:
        try: return m.group(1), json.loads(m.group(2))
        except: return m.group(1), {}
    return None, None

def run_agent(user_input, tools, chat_history=None, verbose=True, max_iter=6):
    # NOTE: This `run_agent` definition is superseded by the one in cell `hcvsulZ9mA2W`.
    # The `NameError` you encountered for `chat` in a later cell is not related to this cell's content.
    # To fix the NameError, ensure cell `fCGltw_CmiIM` (where `chat` is defined) and all preceding cells are executed.
    if chat_history is None: chat_history = []
    tools_map = {t.name: t for t in tools}
    tool_lines = '\n'.join([f'  {t.name}: {t.description.strip().split(chr(10))[0]}' for t in tools])
    system = (f"You are a travel assistant with these tools:\n{tool_lines}\n\n"
              "Call tools as: <function=tool_name{\"param\": \"value\"}></function>\n"
              "When done, write: Final Answer: <your answer>")
    messages = [SystemMessage(content=system)] + list(chat_history) + [HumanMessage(content=user_input)]
    for i in range(max_iter):
        if verbose: print(f"\n🔄 Step {i+1}")
        response = llm.invoke(messages)
        resp = response.content
        messages.append(AIMessage(content=resp))
        if verbose: print(f"💭 {resp[:200]}")
        m = re.search(r'(?i)final\s*answer\s*:\s*(.*)', resp, re.DOTALL)
        if m: return m.group(1).strip(), messages
        name, args = _parse_tool_call(resp)
        if name and name in tools_map:
            if verbose: print(f"🔧 {name}({args})")
            result = str(tools_map[name].invoke(args))[:700]
            if verbose: print(f"📥 {result[:200]}")
            messages.append(HumanMessage(content=f"Tool result ({name}):\n{result}\n\nContinue reasoning."))
        else:
            return resp, messages
    return "Max iterations reached.", messages

print("✅ run_agent() ready")

✅ run_agent() ready


In [ ]:
# ── AGENT SYSTEM MESSAGE ─────────────────────────────────────
# Instead of a ChatPromptTemplate for the agent, we just define the
# system message as a plain string. Our manual loop inserts it as a
# SystemMessage at the top of every call.

TRAVEL_SYSTEM_MSG = (
    'You are a travel planning assistant. '
    'Use the provided tools to answer questions with real-time data. '
    'Always call a tool when you need current or factual information.'
)

print('✅ System message defined.')
print('   Preview:', TRAVEL_SYSTEM_MSG[:80], '...')

✅ System message defined.
   Preview: You are a travel planning assistant. Use the provided tools to answer questions  ...


In [ ]:
# ── THE AGENT LOOP ───────────────────────────────────────────
# This replaces create_tool_calling_agent + AgentExecutor.
# Advantages: no streaming issues, fully transparent, runs on any Groq model.
#
# How it works:
#   1. Bind tools to the LLM so it knows what functions are available
#   2. Call llm_with_tools.invoke(messages)
#   3. If the LLM returns tool_calls → run each tool → append results → repeat
#   4. If no tool_calls → the LLM has a final answer → return it

def run_agent(user_input, tools, chat_history=None, verbose=True, max_iter=5):
    '''
    Manual ReAct-style agent loop using llm.bind_tools().
    Args:
        user_input  : the user question
        tools       : list of @tool-decorated functions
        chat_history: list of previous HumanMessage / AIMessage objects
        verbose     : print tool calls and results
        max_iter    : safety limit on iterations
    Returns:
        (final_answer_str, updated_messages_list)
    '''
    if chat_history is None:
        chat_history = []

    # Map tool names to callables
    tools_map = {t.name: t for t in tools}

    # Build the LLM with tools bound
    llm_with_tools = llm.bind_tools(tools)

    # Assemble starting messages
    messages = (
        [SystemMessage(content=TRAVEL_SYSTEM_MSG)]
        + list(chat_history)
        + [HumanMessage(content=user_input)]
    )

    for iteration in range(max_iter):
        if verbose:
            print(f'\n🔄 Step {iteration + 1}')

        response = llm_with_tools.invoke(messages)   # ← direct invoke, no streaming
        messages.append(response)

        # No tool calls → model has the final answer
        if not response.tool_calls:
            if verbose:
                print('✅ Done — no more tools needed.')
            return response.content, messages

        # Execute every tool the model requested
        for tc in response.tool_calls:
            name = tc['name']
            args = tc['args']
            if verbose:
                print(f'🔧 Tool called : {name}')
                print(f'   Arguments   : {args}')

            result = tools_map[name].invoke(args)

            if verbose:
                preview = str(result)[:200]
                print(f'📥 Tool result : {preview}...' if len(str(result)) > 200 else f'📥 Tool result : {result}')

            messages.append(ToolMessage(
                content=str(result),
                tool_call_id=tc['id']
            ))

    return 'Max iterations reached without a final answer.', messages

print('✅ run_agent() defined — manual tool-calling loop ready.')
print('   Uses: llm.bind_tools()  +  llm_with_tools.invoke()  (no streaming)')

✅ run_agent() defined — manual tool-calling loop ready.
   Uses: llm.bind_tools()  +  llm_with_tools.invoke()  (no streaming)


In [ ]:
# ── TEST QUERY 1: Destination Search ────────────────────────
# Watch the agent call web_search automatically

tools_s1 = [web_search, get_weather]

print('=' * 60)
print('Query 1: Best places in Goa')
print('=' * 60)

answer, _ = run_agent(
    user_input='What are the best places to visit in Goa and when is the ideal time to go?',
    tools=tools_s1,
    verbose=True
)
print()
print('━' * 60)
print('🤖 Final Answer:')
print('━' * 60)
print(answer)

Query 1: Best places in Goa

🔄 Step 1
🔧 Tool called : web_search
   Arguments   : {'query': 'best places to visit in Goa and ideal time to go'}
📥 Tool result : 

🔄 Step 2


/tmp/ipykernel_2129/596202386.py:13: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use time

🔧 Tool called : web_search
   Arguments   : {'query': 'top tourist attractions in Goa India'}
📥 Tool result : - TOP | English meaning - Cambridge Dictionary
  TOP definition: 1. the highest place or part: 2. the flat upper surface of something: 3. in baseball, the first…. Learn more.

- TOP Definition & Meani...

🔄 Step 3


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔧 Tool called : web_search
   Arguments   : {'query': 'best places to visit in Goa India'}
📥 Tool result : 

🔄 Step 4


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


📥 Tool result : 

🔄 Step 5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Done — no more tools needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Final Answer:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Based on general travel information, here is a guide to the best places to visit in Goa and the ideal time to plan your trip:

### **Ideal Time to Visit**
*   **Best Time (November to February):** This is the peak tourist season. The weather is pleasant, sunny, and breezy with temperatures ranging from 20°C to 30°C. It is perfect for beach activities, water sports, and sightseeing.
*   **Summer (March to May):** It gets hot and humid. While fewer tourists visit, it is a good time for budget travelers who don't mind the heat.
*   **Monsoon (June to September):** Goa receives heavy rainfall, turning the landscape lush and green. While water sports are usually suspended and some ferries may stop, it is a great time for nature lovers, spa retreats, and experiencing local culture without the crowds.

### **Best Places to Vi

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── TEST QUERY 2: Weather Check ─────────────────────────────

print('=' * 60)
print('Query 2: Current weather in Shimla')
print('=' * 60)

answer, _ = run_agent(
    user_input='Check the weather in Shimla right now. Is it a good time to visit?',
    tools=tools_s1,
    verbose=True
)
print()
print('━' * 60)
print('🤖 Final Answer:')
print('━' * 60)
print(answer)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Query 2: Current weather in Shimla

🔄 Step 1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔧 Tool called : get_weather
   Arguments   : {'city': 'Shimla'}
📥 Tool result : Weather in Shimla: Light rain, 21.39°C (feels 21.88°C), humidity 88%. Tip: Light layers recommended.

🔄 Step 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Done — no more tools needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Final Answer:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The current weather in Shimla is **21.39°C** with **light rain** and high humidity (88%). It feels like 21.88°C.

**Is it a good time to visit?**
It is a pleasant temperature, but the light rain and high humidity might make outdoor sightseeing slightly damp. It is generally a good time to visit if you don't mind a bit of rain, as the temperature is comfortable.

**Packing Tip:**
Bring light layers and an umbrella or raincoat to stay comfortable.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── DEMONSTRATE THE FORGETTING PROBLEM ──────────────────────

print('🔴 WITHOUT MEMORY — Agent forgets between calls')
print('=' * 60)

# Turn 1: give context
print('Turn 1 — User gives context:')
ans1, _ = run_agent(
    'I want a 4-day trip to Coorg with a budget of 12000 INR.',
    tools=tools_s1, verbose=False
)
print('Agent:', ans1[:200], '...\n')

# Turn 2: no memory passed → agent knows nothing from Turn 1
print('Turn 2 — Follow-up (no history passed):')
ans2, _ = run_agent(
    'What if I extend by 2 more days? Will my budget hold?',
    tools=tools_s1, verbose=False   # ← no chat_history passed
)
print('Agent (forgot context!):', ans2[:200], '...')
print()
print('⚠️  The agent in Turn 2 has no idea about Coorg or the 12000 INR budget.')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔴 WITHOUT MEMORY — Agent forgets between calls
Turn 1 — User gives context:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Agent: Here is a practical **4-day budget itinerary for Coorg** tailored to a **₹12,000 INR** budget. This plan focuses on cost-effective homestays, local transport, and free/low-cost attractions while ensur ...

Turn 2 — Follow-up (no history passed):


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Agent (forgot context!): To give you an accurate answer, I’ll need a few details about your trip:
1. **Destination**: Where are you traveling?
2. **Total Budget**: What’s your overall budget for the trip?
3. **Daily Expenses* ...

⚠️  The agent in Turn 2 has no idea about Coorg or the 12000 INR budget.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── ADDING MEMORY — HOW IT WORKS ────────────────────────────
# With the manual loop, memory is simple:
#   - Keep a chat_history list of HumanMessage + AIMessage pairs
#   - Pass it into run_agent() on every call
#   - run_agent() inserts it between the system message and the new user query
#
# This is exactly what RunnableWithMessageHistory does under the hood.

# Start with an empty history for this session
coorg_history = []

def chat(user_input, history, tools, verbose=False):
    '''
    One conversational turn: run the agent and update history.
    '''
    answer, _ = run_agent(user_input, tools, chat_history=history, verbose=verbose)
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=answer))
    return answer

print('✅ Memory helper ready.')
print('   chat() runs the agent and appends (HumanMessage, AIMessage) to history.')

✅ Memory helper ready.
   chat() runs the agent and appends (HumanMessage, AIMessage) to history.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── TEST MEMORY ACROSS TWO TURNS ────────────────────────────

print('🟢 WITH MEMORY — Agent remembers context')
print('=' * 60)

print('Turn 1:')
ans1 = chat('I want a 4-day trip to Coorg with a budget of 12000 INR.',
            coorg_history, tools_s1)
print('Agent:', ans1[:300], '...\n')

print('Turn 2 — follow-up (agent remembers Coorg + 12000):')
ans2 = chat('What if I extend by 2 more days? Will my budget hold?',
            coorg_history, tools_s1)
print('Agent:', ans2[:300], '...')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🟢 WITH MEMORY — Agent remembers context
Turn 1:


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Agent: Here is a detailed 4-day itinerary for Coorg (Kodagu) tailored to a budget of **12,000 INR**. This budget is comfortable for a solo traveler or a couple if you opt for budget accommodations and local transport.

### **Budget Breakdown (Estimated)**
*   **Accommodation (3 Nights):** ₹3,000 – ₹4,500 ( ...

Turn 2 — follow-up (agent remembers Coorg + 12000):


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

Agent: Extending your trip to **6 days** with the same budget of **12,000 INR** will be **very challenging** and likely impossible without significantly cutting down on comfort.

Here is the breakdown of why:

### **Cost Impact of Adding 2 Days**
Based on the previous budget, adding 2 days would cost appro ...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── INSPECT WHAT IS IN MEMORY ────────────────────────────────

print(f'Messages in coorg_history: {len(coorg_history)}')
for j, msg in enumerate(coorg_history):
    role = '👤 User' if isinstance(msg, HumanMessage) else '🤖 Agent'
    print(f'  [{j+1}] {role}: {str(msg.content)[:80]}...')

Messages in coorg_history: 4
  [1] 👤 User: I want a 4-day trip to Coorg with a budget of 12000 INR....
  [2] 🤖 Agent: Here is a detailed 4-day itinerary for Coorg (Kodagu) tailored to a budget of **...
  [3] 👤 User: What if I extend by 2 more days? Will my budget hold?...
  [4] 🤖 Agent: Extending your trip to **6 days** with the same budget of **12,000 INR** will be...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── TOOL 3: BUDGET CALCULATOR ───────────────────────────────

@tool
def calculate_budget(days: int, daily_budget: float) -> str:
    '''Calculate total trip budget and breakdown by category.
    Args: days — number of trip days; daily_budget — INR per day.
    Returns: total cost with accommodation/food/transport/activities split.
    '''
    total = days * daily_budget
    return (
        f'Budget for {days}-day trip at INR {daily_budget}/day:\n'
        f'  Total         : INR {total:,.0f}\n'
        f'  Accommodation : INR {total*0.40:,.0f} (40%)\n'
        f'  Food          : INR {total*0.25:,.0f} (25%)\n'
        f'  Transport     : INR {total*0.20:,.0f} (20%)\n'
        f'  Activities    : INR {total*0.10:,.0f} (10%)\n'
        f'  Misc          : INR {total*0.05:,.0f}  (5%)\n'
        f'{"✅ Comfortable" if daily_budget >= 3000 else "⚠️ Budget travel — plan carefully"}'
    )

print(f'✅ Tool 3: {calculate_budget.name}')
print(calculate_budget.invoke({'days': 3, 'daily_budget': 2500}))

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Tool 3: calculate_budget
Budget for 3-day trip at INR 2500.0/day:
  Total         : INR 7,500
  Accommodation : INR 3,000 (40%)
  Food          : INR 1,875 (25%)
  Transport     : INR 1,500 (20%)
  Activities    : INR 750 (10%)
  Misc          : INR 375  (5%)
⚠️ Budget travel — plan carefully


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [ ]:
# ── COMPLETE 3-TOOL AGENT WITH MEMORY ───────────────────────

all_tools = [web_search, get_weather, calculate_budget]

# Separate history per 'session'
main_history = []

print('✅ Complete Travel Agent Ready!')
print(f'   Tools  : {[t.name for t in all_tools]}')
print( '   Memory : manual chat_history list')
print(f'   Model  : {GROQ_MODEL}')
print()

# ── COMPOUND QUERY: All 3 Tools in One Turn ──────────────────
print('=' * 65)
print('COMPOUND QUERY — Watch all 3 tools activate')
print('=' * 65)

ans = chat(
    'Plan a 4-day trip to Manali. Check the current weather, '
    'search for the top places to visit, and tell me if '
    '10000 INR is enough budget.',
    main_history, all_tools, verbose=True
)
print()
print('━' * 65)
print('🤖 Travel Agent:')
print('━' * 65)
print(ans)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Complete Travel Agent Ready!
   Tools  : ['web_search', 'get_weather', 'calculate_budget']
   Memory : manual chat_history list
   Model  : qwen/qwen3.6-27b

COMPOUND QUERY — Watch all 3 tools activate

🔄 Step 1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔧 Tool called : get_weather
   Arguments   : {'city': 'Manali'}
📥 Tool result : Weather in Manali: Overcast clouds, 35.28°C (feels 41.76°C), humidity 51%. Tip: Pack light clothes!

🔄 Step 2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

🔧 Tool called : calculate_budget
   Arguments   : {'daily_budget': 2500, 'days': 4}
📥 Tool result : Budget for 4-day trip at INR 2500.0/day:
  Total         : INR 10,000
  Accommodation : INR 4,000 (40%)
  Food          : INR 2,500 (25%)
  Transport     : INR 2,000 (20%)
  Activities    : INR 1,000 ...

🔄 Step 5


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

✅ Done — no more tools needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Travel Agent:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
**Current Weather in Manali:**  
It's currently overcast with a temperature of around **35°C (feels like 42°C)** and 51% humidity. 🌤️ *Packing Tip:* Since it's quite warm, pack light cotton clothes, but always carry a light jacket as mountain weather can change quickly, especially in the evenings.

**Top Places to Visit:**  
Based on current travel trends, here are the must-visit spots in Manali:  
🏔️ **Solang Valley** – Perfect for adventure sports like paragliding, zorbing, and skiing.  
🚗 **Rohtang Pass** – A breathtaking high-altitude mountain pass with snow-capped peaks (permit required).  
🛕 **Hadimba Temple** – A serene ancient temple nestled in a deodar forest.  
🌿 **Old Manali & Mall Road** – Great for cafes, shopping, and experiencing local culture.  
♨️ **Vashisht Hot Springs** – Natural hot water 

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

### ── TOOL 4: CURRENCY CONVERTER ──────────────────────────────

To add a currency converter, we'll need an API key from a currency exchange service (e.g., ExchangeRate-API.com, Open Exchange Rates, Fixer.io). I'll add a placeholder for this new API key (`CURRENCY_API_KEY`).

In [ ]:
CURRENCY_API_KEY = os.getenv('CURRENCY_API_KEY', getpass.getpass('💱 Enter your Currency Exchange API Key: '))

@tool
def currency_converter(amount: float, from_currency: str, to_currency: str) -> str:
    '''Convert an amount from one currency to another using real-time exchange rates.
    Args:
        amount: The amount of money to convert.
        from_currency: The three-letter currency code to convert from (e.g., USD, INR, EUR).
        to_currency: The three-letter currency code to convert to (e.g., USD, INR, EUR).
    Returns: The converted amount and the exchange rate used, or an error message.
    '''
    try:
        # Using ExchangeRate-API as an example. Replace with your chosen API endpoint.
        # Many free tiers offer ~1500 requests/month.
        url = f"https://v6.exchangerate-api.com/v6/{CURRENCY_API_KEY}/pair/{from_currency}/{to_currency}/{amount}"
        response = requests.get(url, timeout=8)
        data = response.json()

        if response.status_code == 200 and data.get('result') == 'success':
            converted_amount = data['conversion_result']
            exchange_rate = data['conversion_rate']
            return (f"{amount} {from_currency} is {converted_amount:.2f} {to_currency}. "
                    f"Exchange Rate: 1 {from_currency} = {exchange_rate:.4f} {to_currency}.")
        elif data.get('result') == 'error' and 'error-type' in data:
            error_type = data['error-type']
            if error_type == 'unsupported-code':
                return (f"Currency conversion error: Invalid currency code provided. "
                        f"Please check '{from_currency}' or '{to_currency}'.")
            elif error_type == 'invalid-key' or error_type == 'malformed-request' or error_type == 'quota-reached':
                 return f"Currency conversion API error: {error_type}. Please check your CURRENCY_API_KEY and API usage limits."
            else:
                 return f"Currency conversion API error: {error_type}."
        else:
            return f"Could not convert currency: {data.get('error-type', 'Unknown error')}"
    except requests.exceptions.RequestException as e:
        return f'Currency conversion request failed: {str(e)}'
    except Exception as e:
        return f'Currency conversion error: {str(e)}'

print(f'✅ Tool 4: {currency_converter.name}')
print(currency_converter.invoke({'amount': 100, 'from_currency': 'USD', 'to_currency': 'INR'}))


💱 Enter your Currency Exchange API Key: ··········
✅ Tool 4: currency_converter
100.0 USD is 9569.50 INR. Exchange Rate: 1 USD = 95.6950 INR.


### ── UPDATE `all_tools` and Test Currency Conversion ──────────────────────────────

Now we'll update the `all_tools` list to include the new `currency_converter` tool and test it with a sample query.

In [ ]:
all_tools = [web_search, get_weather, calculate_budget, currency_converter]

print('✅ Updated all_tools list with currency_converter.')
print(f'   New tools : {[t.name for t in all_tools]}')
print('\n' + '=' * 65)
print('COMPOUND QUERY — Test the new currency converter tool')
print('=' * 65)

ans = chat(
    'What is 50 USD in INR? Also, what is the current weather in New Delhi?',
    main_history, all_tools, verbose=True
)

print('\n' + '━' * 65)
print('🤖 Travel Agent:')
print('━' * 65)
print(ans)


✅ Updated all_tools list with currency_converter.
   New tools : ['web_search', 'get_weather', 'calculate_budget', 'currency_converter']

COMPOUND QUERY — Test the new currency converter tool

🔄 Step 1
🔧 Tool called : currency_converter
   Arguments   : {'amount': 50, 'from_currency': 'USD', 'to_currency': 'INR'}
📥 Tool result : 50.0 USD is 4784.75 INR. Exchange Rate: 1 USD = 95.6950 INR.

🔄 Step 2
🔧 Tool called : get_weather
   Arguments   : {'city': 'New Delhi'}
📥 Tool result : Weather in New Delhi: Overcast clouds, 34.01°C (feels 41.01°C), humidity 59%. Tip: Pack light clothes!

🔄 Step 3
✅ Done — no more tools needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Travel Agent:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
50 USD is approximately **4,784.75 INR** (based on an exchange rate of 1 USD = 95.6950 INR).

The current weather in **New Delhi** is overcast with a temperature of **34.01°C** (feels like 41.01°C) and 59% humidity. **Pack

In [ ]:
# ── FOLLOW-UP: Tests Memory ──────────────────────────────────

# --- FIX for ModuleNotFoundError/ImportError: Ensuring all dependencies are installed and within this cell ---
# This ensures that all necessary packages are installed directly within this cell's execution context.
# Explicitly setting compatible versions to avoid conflicts, especially for langchain-core.
!pip install -q -U \
    "langchain>=0.2.0" \
    "langchain-groq>=0.2.0" \
    "langchain-community>=0.2.0" \
    "langchain-core>=1.3.1" \
    duckduckgo-search \
    requests

# --- 1. Necessary Imports for Tools and Agent --- (originally from various cells)
import os, requests, json, re, warnings, getpass
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_groq import ChatGroq
from duckduckgo_search import DDGS
warnings.filterwarnings('ignore')

# --- 2. API Keys & LLM Setup --- (originally from i5LbStCUkib_ and A9PzQMVFkzuM)
# Assuming keys are already set in environment or provided interactively earlier.
# Using os.getenv for non-interactive retrieval within this self-contained cell.
GROQ_API_KEY = os.getenv('GROQ_API_KEY', getpass.getpass('🔑 Enter your Groq API Key: '))
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY', getpass.getpass('🌤️  Enter your OpenWeatherMap API Key: '))
os.environ['GROQ_API_KEY'] = GROQ_API_KEY # Ensure it's set for ChatGroq constructor

GROQ_MODEL = 'qwen/qwen3.6-27b'
llm = ChatGroq(
        model=GROQ_MODEL,
        temperature=0,
        max_tokens=2048,
        streaming=False,
        api_key=GROQ_API_KEY
    )

# --- 3. Tool Definitions --- (originally from zNxiUi8HlnT8, VwHXTtJplyVl, 3uKzFkrtm9lp)
@tool
def web_search(query: str) -> str:
    '''Search the web for travel destination info, attractions, and tips.
    Args: query — search string.
    Returns: top search results as text.
    '''
    try:
        results = []
        with DDGS() as ddgs:
            for r in list(ddgs.text(query, max_results=3)):
                results.append(f"- {r.get('title','')}\n  {r.get('body','')}")
        output = '\n\n'.join(results)
        return output[:1500] + '...' if len(output) > 1500 else output
    except Exception as e:
        return f'Search unavailable: {str(e)}'

@tool
def get_weather(city: str) -> str:
    '''Get current weather for a city.
    Args: city — city name (e.g. Mumbai, Goa, Shimla).
    Returns: temperature, conditions, humidity, packing tip.
    '''
    try:
        resp = requests.get(
            'http://api.openweathermap.org/data/2.5/weather',
            params={'q': city, 'appid': OPENWEATHER_API_KEY, 'units': 'metric'},
            timeout=8
        )
        d = resp.json()
        if d.get('cod') == 200:
            cond  = d['weather'][0]['description'].capitalize()
            temp  = d['main']['temp']
            feels = d['main']['feels_like']
            hum   = d['main']['humidity']
            tip   = ('Pack light clothes!' if temp > 30 else
                     'Light layers recommended.' if temp > 20 else
                     'Carry a jacket.' if temp > 10 else
                     'Pack warm clothes!')
            return (f'Weather in {city}: {cond}, {temp}°C '
                    f'(feels {feels}°C), humidity {hum}%. Tip: {tip}')
        return f'Could not get weather for {city}. Check the city name.'
    except Exception as e:
        return f'Weather error: {str(e)}'

@tool
def calculate_budget(days: int, daily_budget: float) -> str:
    '''Calculate total trip budget and breakdown by category.
    Args: days — number of trip days; daily_budget — INR per day.
    Returns: total cost with accommodation/food/transport/activities split.
    '''
    total = days * daily_budget
    return (
        f'Budget for {days}-day trip at INR {daily_budget}/day:\n'
        f'  Total         : INR {total:,.0f}\n'
        f'  Accommodation : INR {total*0.40:,.0f} (40%)\n'
        f'  Food          : INR {total*0.25:,.0f} (25%)\n'
        f'  Transport     : INR {total*0.20:,.0f} (20%)\n'
        f'  Activities    : INR {total*0.10:,.0f} (10%)\n'
        f'  Misc          : INR {total*0.05:,.0f}  (5%)\n'
        f'{"✅ Comfortable" if daily_budget >= 3000 else "⚠️ Budget travel — plan carefully"}'
    )

# --- 4. Agent System Message --- (originally from c6f-n4tIl9c8)
TRAVEL_SYSTEM_MSG = (
    'You are a travel planning assistant. '
    'Use the provided tools to answer questions with real-time data. '
    'Always call a tool when you need current or factual information.'
)

# --- 5. run_agent function --- (originally from hcvsulZ9mA2W)
def _parse_tool_call(text):
    m = re.search(r'<function=(\w+)({".*?"})\s*(?:</function>)?', text, re.DOTALL)
    if m:
        try: return m.group(1), json.loads(m.group(2))
        except: return m.group(1), {}
    return None, None

def run_agent(user_input, tools, chat_history=None, verbose=True, max_iter=5):
    if chat_history is None:
        chat_history = []
    tools_map = {t.name: t for t in tools}
    llm_with_tools = llm.bind_tools(tools)
    messages = ([SystemMessage(content=TRAVEL_SYSTEM_MSG)] + list(chat_history) + [HumanMessage(content=user_input)])
    for iteration in range(max_iter):
        if verbose: print(f'\n🔄 Step {iteration + 1}')
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            if verbose: print('✅ Done — no more tools needed.')
            return response.content, messages
        for tc in response.tool_calls:
            name = tc['name']
            args = tc['args']
            if verbose: print(f'🔧 Tool called : {name}\n   Arguments   : {args}')
            result = tools_map[name].invoke(args)
            if verbose:
                preview = str(result)[:200]
                print(f'📥 Tool result : {preview}...' if len(str(result)) > 200 else f'📥 Tool result : {result}')
            messages.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
    return 'Max iterations reached without a final answer.', messages

# --- 6. Initialize all_tools and main_history and chat function --- (previously from W1mNEqphnNHC and fCGltw_CmiIM)
all_tools = [web_search, get_weather, calculate_budget]
main_history = []

def chat(user_input, history, tools, verbose=False):
    '''
    One conversational turn: run the agent and update history.
    '''
    answer, _ = run_agent(user_input, tools, chat_history=history, verbose=verbose)
    history.append(HumanMessage(content=user_input))
    history.append(AIMessage(content=answer))
    return answer

# --- END FIX ---

print('=' * 65)
print('FOLLOW-UP — Agent should remember Manali + 10000 INR')
print('=' * 65)

ans2 = chat(
    'What if I extend by 2 more days? Recalculate the budget.',
    main_history, all_tools, verbose=True
)
print()
print('━' * 65)
print('🤖 Agent (remembers context):')
print('━' * 65)
print(ans2)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.0/147.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 18.5 MB/s eta 0:00:00
🔑 Enter your Groq API Key: ··········
🌤️  Enter your OpenWeatherMap API Key: ··········
FOLLOW-UP — Agent should remember Manali + 10000 INR

🔄 Step 1
✅ Done — no more tools needed.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🤖 Agent (remembers context):
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
To recalculate your budget with the 2 extra days, I'll need a couple of details:
- What was your **original trip duration** (in days)?
- What is your **daily budget** (in INR)?

Once you share those, I'll run the updated budget breakdown for you right away!


In [ ]:
# ── PRODUCTION-READY WRAPPER ─────────────────────────────────

def safe_travel_agent(query: str, history: list = None, verbose: bool = False) -> str:
    '''
    Production-ready travel agent with full error handling.
    '''
    if not query or not query.strip():
        return '❌ Please enter a travel question.'
    if len(query) > 2000:
        return '❌ Query too long. Keep it under 2000 characters.'
    if history is None:
        history = []
    try:
        answer, _ = run_agent(query.strip(), all_tools, chat_history=history, verbose=verbose)
        return answer
    except Exception as e:
        err = str(e).lower()
        if 'rate limit' in err or '429' in err:
            return '⏳ Rate limit hit. Wait 30 seconds and retry.'
        elif 'api key' in err or '401' in err:
            return '🔑 API key error. Check your GROQ_API_KEY.'
        elif 'timeout' in err:
            return '⏱️ Request timed out. Try again.'
        return f'❌ Error: {str(e)}'

print('✅ safe_travel_agent() defined.')

✅ safe_travel_agent() defined.


In [ ]:
# ── TEST THE SAFE WRAPPER ────────────────────────────────────

result = safe_travel_agent('What is the current weather in Chennai?')
print('✅ Normal query:')
print(result)
print()
print('✅ Empty query handled:', safe_travel_agent(''))


✅ Normal query:
The current weather in Chennai is overcast with a temperature of **35.55°C** (feels like 42.43°C). The humidity is at 51%.

**Packing Tip:** Pack light clothes!

✅ Empty query handled: ❌ Please enter a travel question.
